# 03 · Artefacto final: narrativa interactiva con Plotly Graph Objects
**En una frase:** convertimos los tres hallazgos en una página interactiva que cuenta la historia de principio a fin y se abre sin instalar nada.

**Detalle técnico:** pasos 5, 6 y 7 de la guía. Pasamos del modo exploratorio al **explicativo**: las tres perlas se ensamblan en un HTML autónomo de tres actos, con el arco **Bing → Bang → Bongo** (anunciar el mensaje, mostrar la evidencia y repetir el mensaje junto con la acción).

| Acto | Pregunta | Perla | Interactividad (y por qué) |
|---|---|---|---|
| 1 · Planteamiento | ¿Qué está cambiando? | Índice 2018–2026 (líneas) | Menú *Comparar con…*: permite auditar la afirmación contra cualquier otro delito; el título y las etiquetas se reescriben. |
| 2 · Nudo | ¿Dónde? | Ranking de localidades (barras horizontales, base 0) | Botones de métrica: aumento absoluto, volumen y %; el color sigue a las mismas 5 localidades. |
| 3 · Desenlace | ¿Cuándo actuar? | Mapa de calor localidad × franja | Solo *hover* (detalles bajo demanda). No se añaden controles porque no aportarían. |
| Síntesis | ¿Qué cambió, en conjunto? | Tres paneles de mancuernas: promedio 2018–2022 → 2026 | Solo *hover*: la figura resume, no explora. |

Todas las cifras de títulos, anotaciones, HTML y PDF salen de **un solo diccionario (`kpis`)** calculado desde los datos; ninguna se escribe a mano.

In [1]:
%load_ext autoreload
%autoreload 2
# Configuración del entorno: el notebook vive en /notebooks, el código en /src
import sys, pathlib
RAIZ = pathlib.Path.cwd().resolve()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
print("Raíz del proyecto:", RAIZ)

Raíz del proyecto: C:\Users\ADRIAN\Desktop\Ciencia de datos\Visualizacion de datos\Proyecto_Plotly_SDSCJ


In [2]:
import json
from src import normalizacion as N, analisis as A, visual as V, artefacto as ART, config as C

df = N.cargar()
reporte = json.loads(C.ARCHIVO_CALIDAD.read_text(encoding="utf-8"))
kpis = A.construir_kpis(df, reporte)
ART.guardar_kpis(kpis)
print(json.dumps({k: kpis[k] for k in ["ventana", "foco", "contraste",  "pct_aumento_top5"]},
                 indent=2, ensure_ascii=False))

{
  "ventana": "enero–septiembre",
  "foco": {
    "delito": "Violencia intrafamiliar",
    "base": 26767.6,
    "actual": 43013,
    "cambio_pct": 60.7,
    "cambio_abs": 16245.4,
    "racha": 3,
    "participacion_pct": 24.8,
    "participacion_base_pct": 17.5,
    "ranking_cambio_abs": 1,
    "casos_2022": 25719,
    "cambio_pct_vs_2022": 67.2
  },
  "contraste": {
    "delito": "Hurto a personas",
    "actual": 86376,
    "cambio_pct_vs_base": 7.8,
    "anio_pico": 2023,
    "pico": 120489,
    "cambio_pct_vs_pico": -28.3
  },
  "pct_aumento_top5": 45.7
}


## Verificación de las afirmaciones antes de graficar
**En una frase:** si un título dice algo que los datos ya no sostienen, el código se detiene y avisa.

Un título activo es una afirmación. Estas pruebas (`assert`) comprueban cada una antes de graficar, para proteger la honestidad narrativa (Cairo, 2019).

In [3]:
f = kpis["foco"]
assert f["ranking_cambio_abs"] == 1, "La VIF ya no es el mayor aumento absoluto: revisar el titular del Acto 1"
assert f["cambio_pct"] > 0 and f["cambio_pct_vs_2022"] > 0, "La VIF no supera su base ni 2022: revisar la Gran Idea"
assert kpis["contraste"]["cambio_pct_vs_pico"] < 0, "El hurto no retrocede: revisar la Gran Idea"
assert kpis["perfil"]["pct_noche_madrugada"] > 50, "Noche + madrugada ya no son la mayoría: revisar el Acto 3"
print("✔ Las afirmaciones del Bing, el Bang y el Bongo se sostienen con los datos.")
A.resumen_delitos(A.tabla_anual(A.ventana_comparable(df)[0]), kpis["ultimo_anio"])

✔ Las afirmaciones del Bing, el Bang y el Bongo se sostienen con los datos.


,base_2018_2022,casos_2026,cambio_abs,cambio_pct,anios_consecutivos_al_alza,participacion_pct
Violencia intrafamiliar,26767.6,43013,16245.4,60.7,3,24.8
Hurto a personas,80089.4,86376,6286.6,7.8,0,49.8
Delitos sexuales,4460.2,6613,2152.8,48.3,0,3.8
Lesiones personales,16544.8,17636,1091.2,6.6,2,10.2
Extorsión,842.2,1639,796.8,94.6,0,0.9
Hurto de motocicletas,3231.6,3392,160.4,5.0,0,2.0
Homicidios,776.2,819,42.8,5.5,0,0.5
Secuestro,7.6,23,15.4,202.6,0,0.0
Hurto a entidades financieras,31.0,4,-27.0,-87.1,0,0.0
Hurto de automotores,2668.0,2297,-371.0,-13.9,0,1.3


## Perla 1 · Acto 1 — Planteamiento (Bing)
**En una frase:** muestra que la violencia intrafamiliar es el delito que más casos suma sobre su promedio, mientras el hurto baja.

* **Gráfico:** líneas, porque la intención es mostrar una **tendencia** (Sesión 6). Se usa un índice (promedio 2018–2022 = 100) para comparar en la misma escala delitos con volúmenes muy distintos; la posición sobre una escala común es el canal visual más preciso (Munzner, 2014).
* **Atributos preatentivos:** línea coral gruesa para la VIF, gris pizarra para el hurto (el contraste) y gris claro y fino para los otros 8 delitos.
* **Anotaciones:** etiqueta al final de cada línea en lugar de leyenda, banda de pandemia (región común) y una nota que explica 2023.
* **Ética:** 2026 se dibuja con punto hueco porque septiembre es preliminar; se excluyen los delitos con menos de 100 casos al año.
* **Titular robusto:** no depende de 2023 (año con cambios de registro); la anotación muestra también la comparación con 2022 (+67 %).
* **Evidencia y decisión:** debajo del gráfico, un bloque breve con la evidencia y la decisión (el mismo del HTML); el titular no se repite.

In [4]:
fig1 = V.figura_indice(df, kpis)
fig1.show(config=V.CONFIG_PLOTLY)

In [5]:
# Evidencia y decisión (el mismo bloque que aparece en el HTML)
from IPython.display import HTML
HTML(ART.historias(df, kpis)[0])

## Perla 2 · Acto 2 — Nudo (Bang)
**En una frase:** muestra en qué localidades se concentra el aumento.

* **Gráfico:** barras horizontales **ordenadas** (ranking) con el eje en **cero**; los nombres se leen sin girar etiquetas.
* **Color:** las 5 localidades con mayor aumento en coral y el resto en gris. El color se mantiene al cambiar de métrica, para que el ojo pueda seguirlas.
* **Interactividad:** tres botones con `method="update"` que cambian a la vez barras, título, anotaciones y rango del eje.
* **Evidencia y decisión:** debajo del gráfico, un bloque breve con la evidencia y la decisión (el mismo del HTML); el titular no se repite.

In [6]:
fig2 = V.figura_localidades(df, kpis)
fig2.show(config=V.CONFIG_PLOTLY)

In [7]:
# Evidencia y decisión (el mismo bloque que aparece en el HTML)
from IPython.display import HTML
HTML(ART.historias(df, kpis)[1])

## Perla 3 · Acto 3 — Desenlace (Bongo)
**En una frase:** muestra que noche y madrugada concentran 6 de cada 10 casos, justo cuando casi todas las comisarías están cerradas.

* **Fuente externa:** el horario de las comisarías de familia (23 cierran a las 4:00 p. m., 5 a las 11:00 p. m. y solo 2 atienden 24 horas) proviene del Portal Bogotá (Grandas Medina, 2025) y está en `src/config.py` (`COMISARIAS`).

* **Gráfico:** mapa de calor con paleta **secuencial de un solo matiz** que arranca en 0 (más intenso = más casos) y el valor escrito en cada celda, para no depender solo del color.
* **Orden:** franjas en orden cronológico y localidades de mayor a menor volumen; las 5 del Acto 2 se resaltan en el eje para mantener la continuidad de la historia.
* **Evidencia y decisión:** debajo del gráfico, un bloque breve con la evidencia y la decisión (el mismo del HTML); el titular no se repite.

In [8]:
fig3 = V.figura_franjas(df, kpis)
fig3.show(config=V.CONFIG_PLOTLY)

In [9]:
# Evidencia y decisión (el mismo bloque que aparece en el HTML)
from IPython.display import HTML
HTML(ART.historias(df, kpis)[2])

## Síntesis — el cambio en una sola figura
**En una frase:** reúne en una sola mirada qué cambió, dónde y cuándo, para que la audiencia se lleve la idea completa.

* **Gráfico:** tres paneles de **mancuernas** con la misma gramática: ○ promedio 2018–2022 → ● 2026. El cambio se lee por **posición** sobre una escala común (Munzner, 2014) y la línea une el antes con el después (**conexión**, Gestalt).
* **Generaliza los tres actos:** delitos de alto impacto (Acto 1), las 8 localidades con mayor aumento (Acto 2) y las cuatro franjas del día (Acto 3).
* **Color:** coral solo en lo que la historia pide mirar (la VIF, las cinco localidades priorizadas y las dos franjas que más crecen).
* **Conclusiones:** cada panel trae su conclusión debajo, calculada desde los datos.

In [10]:
fig4 = V.figura_sintesis(df, kpis)
fig4.show(config=V.CONFIG_PLOTLY)

## Exportación a HTML autónomo (paso 7)
**En una frase:** el resultado es un solo archivo que se abre con doble clic, sin Python y sin internet.

`include_plotlyjs=True` incrusta el motor de Plotly **una sola vez** dentro del archivo (criterio *Estratégico* de la rúbrica: "HTML totalmente autónomo"). Además se simplifica la barra de herramientas (sin logo, lazo ni selección) para que un usuario no técnico no se pierda.

In [11]:
ruta_html = ART.construir_html(df, kpis)

✔ HTML autónomo: C:\Users\ADRIAN\Desktop\Ciencia de datos\Visualizacion de datos\Proyecto_Plotly_SDSCJ\salidas\Violencia_intrafamiliar_Bogota_SDSCJ.html (4.9 MB)


## Control de calidad del entregable

In [12]:
html = C.ARCHIVO_HTML.read_text(encoding="utf-8")
import re
import numpy as np
def n_puntos(fig):
    return int(sum(np.asarray(t.z).size if getattr(t, "z", None) is not None else len(t.x) for t in fig.data))
puntos = {f"Perla {i}": n_puntos(fig) for i, fig in enumerate([fig1, fig2, fig3, fig4], start=1)}
chequeos = {
    "Archivo HTML existe": C.ARCHIVO_HTML.exists(),
    "Tamaño (MB)": round(C.ARCHIVO_HTML.stat().st_size / 1e6, 2),
    "plotly.js incrustado (no depende de CDN)": '<script src="https://cdn.plot.ly' not in html and html.count("plotly.js v") >= 1,
    "Tres perlas + síntesis en el HTML": all(f'id="perla{i}"' in html for i in (1, 2, 3, 4)),
    "Puntos de datos por figura (latencia < 0,1 s)": puntos,
    # Solo enlaces reales (src=/href=); plotly.js trae "file://" como texto interno
    "Sin enlaces a archivos locales": not re.search(r"(src|href)=[\"']file://", html),
}
for k, v in chequeos.items():
    print(f"{'✔' if v else '✘'} {k}: {v}")

✔ Archivo HTML existe: True
✔ Tamaño (MB): 4.94
✔ plotly.js incrustado (no depende de CDN): True
✔ Tres perlas + síntesis en el HTML: True
✔ Puntos de datos por figura (latencia < 0,1 s): {'Perla 1': 90, 'Perla 2': 60, 'Perla 3': 76, 'Perla 4': 72}
✔ Sin enlaces a archivos locales: True


> **Latencia:** cada figura dibuja menos de 300 puntos y los controles solo cambian la visibilidad y los colores de trazas ya cargadas (`update`), sin recalcular datos. Medido con `Plotly.update` en Chromium: entre 25 y 45 ms por actualización (por debajo de 0,1 s). También se probó sin conexión: las 3 figuras se dibujan y no hay peticiones externas.

## Documento de justificación (PDF, APA 7) y paquete .zip (paso 8)

In [13]:
import subprocess, sys
subprocess.run([sys.executable, str(C.RAIZ / "docs" / "generar_justificacion.py")], check=True)
print("✔", C.ARCHIVO_PDF.relative_to(C.RAIZ))

✔ docs\Documento_justificacion_APA7.pdf


In [14]:
from src.empaquetar import empaquetar
zip_path = empaquetar()
zip_path

✔ ZIP de entrega: C:\Users\ADRIAN\Desktop\Ciencia de datos\Visualizacion de datos\Proyecto_Plotly_SDSCJ\entrega\Proyecto_Plotly_SDSCJ_entrega.zip (5.8 MB)


WindowsPath('C:/Users/ADRIAN/Desktop/Ciencia de datos/Visualizacion de datos/Proyecto_Plotly_SDSCJ/entrega/Proyecto_Plotly_SDSCJ_entrega.zip')

### Productos del proyecto
| Producto | Archivo |
|---|---|
| Visualización interactiva (HTML autónomo) | `salidas/Violencia_intrafamiliar_Bogota_SDSCJ.html` |
| Documento de justificación (PDF, APA 7) | `docs/Documento_justificacion_APA7.pdf` |
| Paquete de entrega (.zip) | `entrega/Proyecto_Plotly_SDSCJ_entrega.zip` |

**Integrantes:** Adrian Felipe Grateron, Jhonatan Smith Doncel y Jorge Emmanuel Marin.